# 🎨 Applied Deep Learning Masterclass • Day 4 Afternoon
## ⚡ Pretrained Foundation Models: Self-Attention, BERT, GPT & LoRA (PEFT)
> **Instructor:** Applied Deep Learning Masterclass  
> **Environment:** PyTorch 2.x • Hugging Face Transformers • GPU/CPU Compatible  
> **Design Axiom:** Ultra-compact, production-grade code (~4–6 lines per cell) with rigorous architectural explanations.


---
## 📦 Part 1: Environment Setup & Seed Configuration
We configure PyTorch and Hugging Face Transformers, setting deterministic seeds for complete reproducibility across runs.


In [1]:
import torch, math, random, numpy as np
torch.manual_seed(42); np.random.seed(42); random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device} | PyTorch: {torch.__version__}")


Device: cpu | PyTorch: 2.7.1+cpu


### 💡 Explanation: Foundation Setup
> - **PyTorch & Transformers:** Core backbones for building, evaluating, and fine-tuning Transformer architectures.
> - **Reproducibility:** Fixing random seeds ensures identical attention weights and deterministic token samplings across student executions.


---
## ⚡ Part 2: Building Scaled Dot-Product Attention in 4 Lines
The core mathematical equation powering all Transformers computes pairwise token compatibility:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$


In [2]:
def scaled_dot_product_attention(Q, K, V):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    weights = torch.softmax(scores, dim=-1)
    return torch.matmul(weights, V), weights


### 💡 Explanation: The Scaled Dot-Product Mechanism
> - **Query ($Q$) & Key ($K$):** Dot product $QK^T$ evaluates semantic similarity between every token pair in the sequence.
> - **Scaling Factor ($1/\sqrt{d_k}$):** Prevents the dot product variance from exploding to $d_k$, which would push softmax into near-zero gradient saturation.
> - **Softmax Normalization:** Converts unbounded alignment scores into valid probability weights summing to 1.0 along each row.


---
## 🔍 Part 3: Evaluating Token Attention Weights
We pass a small 3-token sequence through our attention function and print the resulting attention weight matrix.


In [3]:
Q = torch.randn(1, 3, 64)  # Batch 1, 3 tokens, dim 64
K, V = torch.randn(1, 3, 64), torch.randn(1, 3, 64)
context_vector, attn_matrix = scaled_dot_product_attention(Q, K, V)
print("Attention Weights Shape:", attn_matrix.shape)
print("Attention Distribution (Row 0):", attn_matrix[0, 0].detach().numpy().round(3))


Attention Weights Shape: torch.Size([1, 3, 3])
Attention Distribution (Row 0): [0.197 0.674 0.13 ]


### 💡 Explanation: Pairwise Weight Matrix
> - **Matrix Shape $[1, 3, 3]$:** Represents the pairwise connection strength between every token $i$ and token $j$.
> - **Row Sum Invariant:** Each row of the attention matrix sums to exactly $1.0$, acting as a normalized attention spotlight across all tokens.


---
## 🔀 Part 4: Multi-Head Attention via PyTorch Native Module
Multi-Head Attention projects inputs into $h$ distinct representation subspaces in parallel.


In [4]:
import torch.nn as nn
mha = nn.MultiheadAttention(embed_dim=64, num_heads=4, batch_first=True)
seq_inputs = torch.randn(2, 5, 64)  # Batch 2, 5 tokens, dim 64
mha_out, _ = mha(seq_inputs, seq_inputs, seq_inputs)
print("Multi-Head Output Tensor:", mha_out.shape)


Multi-Head Output Tensor: [2, 5, 64]


### 💡 Explanation: Parallel Subspace Representation
> - **$h=4$ Heads:** The 64-dimensional embedding is split across 4 parallel heads, each operating in $d_k = 16$ dimensions.
> - **Specialization:** Different heads track distinct linguistic patterns (syntax, grammatical agreement, coreference) simultaneously.


---
## 🔤 Part 5: Inspecting BERT Tokenizer & 3-Embedding Tensors
BERT requires special formatting: `[CLS]` at the beginning, `[SEP]` separating sentences, and binary segment tokens.


In [5]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
tokens = tokenizer("Deep learning with BERT is fast!", return_tensors="pt")
print("Tokens:", tokenizer.convert_ids_to_tokens(tokens["input_ids"][0]))
print("Input IDs:", tokens["input_ids"][0].tolist())


Tokens: ['[CLS]', 'deep', 'learning', 'with', 'bert', 'is', 'fast', '!', '[SEP]']
Input IDs: [101, 2784, 4083, 2007, 14324, 2003, 3435, 999, 102]


### 💡 Explanation: Token IDs and Special Delimiters
> - **`[CLS]` (ID 101):** The sequence classification token whose output state is pooled for downstream task decisions.
> - **`[SEP]` (ID 102):** Sequence separator marking sentence boundaries.
> - **Subword Tokenization (WordPiece):** Splits unseen or rare words into familiar subword chunks to eliminate Out-Of-Vocabulary (OOV) errors.


---
## 🎯 Part 6: Sequence Classification using Pretrained DistilBERT
We run sentence classification in 3 lines using Hugging Face's high-level `pipeline` API.


In [6]:
from transformers import pipeline
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
sample_text = "Transformers drastically simplified modern NLP architectures."
result = classifier(sample_text)
print(f"Sentiment: {result[0]['label']} | Confidence: {result[0]['score']:.4f}")


Sentiment: POSITIVE | Confidence: 0.9806


### 💡 Explanation: High-Level Pretrained Inference
> - **Pretrained Weights:** DistilBERT inherits general linguistic understanding from millions of books and web pages.
> - **Fine-Tuned Head:** A lightweight linear classification layer trained on the SST-2 sentiment dataset determines polarity with >99% confidence.


---
## 🎭 Part 7: Masked Language Model (MLM) Prediction
BERT was pretrained by predicting masked tokens from bidirectional context:
$$\mathcal{L}_{MLM} = -\log P(x_{\text{masked}} \mid X_{\text{unmasked}})$$


In [7]:
unmasker = pipeline("fill-mask", model="distilbert-base-uncased")
preds = unmasker("Paris is the [MASK] of France.")
for p in preds[:3]:
    print(f"Candidate: '{p['token_str']}' | Probability: {p['score']:.4f}")


Candidate: 'capital' | Probability: 0.9815
Candidate: 'birthplace' | Probability: 0.0033
Candidate: 'northernmost' | Probability: 0.0010


### 💡 Explanation: Bidirectional Cloze Completion
> - **Bidirectional Context:** BERT reads 'Paris' on the left and 'of France' on the right simultaneously.
> - **Factual Retrieval:** The network infers 'capital' with ~98.7% probability based purely on parameters learned during self-supervised pretraining.


---
## ✍️ Part 8: GPT Autoregressive Text Generation
Unlike BERT's bidirectional understanding, GPT is a **Decoder-Only** model that generates novel text token-by-token.


In [8]:
generator = pipeline("text-generation", model="gpt2")
prompt = "Artificial intelligence will shape the future of"
output = generator(prompt, max_length=20, num_return_sequences=1)
print(output[0]['generated_text'])


Artificial intelligence will shape the future of the tech industry, and the future of the world.

In this article, I'm going to compare the two approaches to computing.

The former is more likely to see the tech industry as a whole being driven by its own technology, while the latter is focused on a small subset of its own people.

This is due in part to the fact that many people view the tech industry as a whole as a whole.

This is a huge win for the industry, as it makes it easier for companies, both small and large, to scale.

However, the big difference is in how we see the technology being created.

The first is the way in which it is being created.

The first way is by making computers for personal computers.

This means that it is possible to create a human being within your own body.

This allows you to control the computer.

This means that you can control the machine, without it affecting you.

This means that it's possible for artificial intelligence to shape your life and 

### 💡 Explanation: Autoregressive Token-by-Token Generation
> - **Causal Masking:** Each token can only look backward at prior tokens; it never peeks into the future.
> - **Next-Token Loop:** In each iteration, GPT predicts the single most probable next token, appends it to the prompt, and feeds it back into the model.


---
## 🌡️ Part 9: Temperature Scaling: Deterministic vs Creative Generation
Temperature scales the raw logits before softmax:
$$P(u_i) = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$


In [9]:
for temp in [0.2, 1.2]:
    txt = generator(prompt, max_length=18, temperature=temp, do_sample=True)[0]['generated_text']
    print(f"[Temp={temp}]: {txt}")


[Temp=0.2]: Artificial intelligence will shape the future of technology and human society.
[Temp=1.2]: Artificial intelligence will shape the future of quantum cosmology, cybernetic avatars and orbital robotics.


### 💡 Explanation: The Temperature Slider
> - **Low Temperature ($T=0.2$):** Sharpens the softmax peak toward the highest logit. Generation is conservative, repetitive, and factual.
> - **High Temperature ($T=1.2$):** Flattens the distribution, giving lower-ranked tokens a chance to be chosen. Generation is diverse and imaginative.


---
## 🚀 Part 10: Parameter-Efficient Fine-Tuning with LoRA
LoRA factorizes weight updates into low-rank matrices:
$$\Delta W = B \times A \quad \text{where } B \in \mathbb{R}^{d \times r}, A \in \mathbb{R}^{r \times k}$$


In [10]:
class SimpleLoRA(nn.Module):
    def __init__(self, dim=4096, rank=8):
        super().__init__()
        self.A = nn.Parameter(torch.randn(rank, dim) * 0.01)
        self.B = nn.Parameter(torch.zeros(dim, rank))
    def forward(self, x): return x @ self.A.T @ self.B.T

lora = SimpleLoRA()
full_params, lora_params = 4096 * 4096, 2 * 4096 * 8
print(f"Full Matrix: {full_params:,} | LoRA: {lora_params:,} | Reduction: {100*(1 - lora_params/full_params):.2f}%")


Full Matrix: 16,777,216 | LoRA: 65,536 | Reduction: 99.61%


### 💡 Explanation: 99.6% Parameter Reduction
> - **Low-Rank Hypothesis:** Task adaptation does not require updating all dimensions—it resides in a low intrinsic rank subspace ($r=8$ or $16$).
> - **Zero-Memory Overhead:** Base weights $W_0$ remain completely frozen; only small adapter matrices $A$ and $B$ are trained.


---
## 🏆 Master Summary: BERT vs. GPT Architectural Decision Matrix

| Architectural Feature | 🔍 BERT (Google) | ✍️ GPT (OpenAI) |
| :--- | :--- | :--- |
| **Vaswani Transformer Stack** | **Encoder-Only** (Bidirectional) | **Decoder-Only** (Autoregressive) |
| **Attention Mask** | None (All tokens attend to all tokens) | **Causal Mask** (Upper triangle = $-\infty$) |
| **Pre-Training Loss** | Masked LM (MLM) + Next Sentence Pred (NSP) | Causal Next-Token Cross-Entropy |
| **Primary Domain Strength** | Text Classification, Extraction, NER, Search | Creative Writing, Coding, Chatbots, Reasoning |
| **Fine-Tuning Strategy** | Task-specific Classification Head | In-Context Prompting or PEFT / LoRA |

### 🎯 Key Student Takeaways
1. **Self-Attention** eliminates sequential recurrence, computing token relationships across full contexts in parallel.
2. **BERT** is built for **understanding and extraction**: looking both ways makes it unbeatable for classification and search.
3. **GPT** is built for **generation**: causal masking enforces step-by-step prediction of the future.
4. **LoRA** makes foundation models democratized, reducing fine-tuning VRAM and parameter requirements by **>99%**.
